# Class 3 — Building a Simple RAG Chatbot

**Week 6: Foundations of RAG and Chatbots**

### Learning objectives
By the end of this notebook you will be able to:
- Chunk a small set of documents into retrieval-sized pieces
- Embed chunks and store them in a simple in-memory structure
- Retrieve the top-k most relevant chunks for a question using cosine similarity
- Construct a grounded prompt from retrieved chunks and generate a final answer
- Name the main limitations of this simple pipeline and what production systems add on top

> This is the capstone notebook for Week 6: everything from Class 1 (grounding) and Class 2 (embeddings and
> similarity) comes together into one working, minimal RAG chatbot.
>
> After you finish, `class-3-langchain.ipynb` rebuilds the same pipeline with LangChain, Chroma, and FAISS.

## Setup

Install Groq (for generation) and `sentence-transformers` (for embeddings). Never hardcode a key in a notebook.

```bash
export GROQ_API_KEY="gsk-..."
```

In Colab: add a secret named `GROQ_API_KEY` and enable notebook access.
This notebook embeds locally with `all-MiniLM-L6-v2` and generates with Groq (`llama-3.3-70b-versatile`).

In [1]:
!pip install -q groq sentence-transformers

import os
import numpy as np
from sentence_transformers import SentenceTransformer

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
except Exception:
    pass

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
_embedder = SentenceTransformer(EMBEDDING_MODEL)
print(f"Loaded local embedding model: {EMBEDDING_MODEL}")

if not GROQ_API_KEY:
    print(
        "No GROQ_API_KEY found.\n"
        "Set it in your environment or add a Colab secret named GROQ_API_KEY.\n"
        "Retrieval still runs locally. Generation cells below will skip until a key is available."
    )
else:
    print("Found GROQ_API_KEY. Retrieval and generation demos are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.4 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded local embedding model: all-MiniLM-L6-v2
Found GROQ_API_KEY. Retrieval and generation demos are ready.


In [3]:
def call_llm(prompt, system=None, model="openai/gpt-oss-120b"):
    """Send a single prompt to Groq. Returns assistant text, or None if no key."""
    if not GROQ_API_KEY:
        print("Skipping live call — no GROQ_API_KEY set.")
        return None
    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY)
    messages = [
        {"role": "system", "content": system or "You are a helpful, concise assistant."},
        {"role": "user", "content": prompt},
    ]
    response = client.chat.completions.create(
        model=model,
        max_tokens=400,
        messages=messages,
    )
    return response.choices[0].message.content


def get_embedding(text, model=EMBEDDING_MODEL):
    """Return the embedding vector (a list of floats) for a piece of text."""
    return _embedder.encode(text).tolist()


def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## Concept 1 — Our Tiny Document Set

A real RAG system starts with a pile of documents (PDFs, wiki pages, tickets). Ours is three short plain-text
"policy" documents, kept intentionally small so the whole pipeline is easy to trace end to end.

In [4]:
raw_documents = [
    """Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting
    from their first day. Unused days roll over up to a maximum of 5 days into the next year.""",

    """Expense Policy: Employees must submit expense reports within 30 days of purchase using the
    finance portal. Reports missing a receipt over $25 will be returned for correction.""",

    """Remote Work Policy: Remote employees must be reachable by chat or phone between 10am and 3pm
    in their local time zone, and are expected to attend the weekly all-hands meeting on video.""",
]

for i, doc in enumerate(raw_documents, start=1):
    print(f"--- Document {i} ---")
    print(doc.strip())
    print()

--- Document 1 ---
Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting
    from their first day. Unused days roll over up to a maximum of 5 days into the next year.

--- Document 2 ---
Expense Policy: Employees must submit expense reports within 30 days of purchase using the
    finance portal. Reports missing a receipt over $25 will be returned for correction.

--- Document 3 ---
Remote Work Policy: Remote employees must be reachable by chat or phone between 10am and 3pm
    in their local time zone, and are expected to attend the weekly all-hands meeting on video.



## Concept 2 — Chunking

Our documents are already short, so each one becomes a single chunk. For a longer document you would split it
into paragraph-sized pieces first (see Class 3's slide deck for chunking strategies), then treat every chunk the
same way we're about to treat these three documents. We'll still write `chunk_documents` as a real, general
function so this notebook works if you paste in longer documents later.

In [5]:
def chunk_text(text, max_chars=280, overlap=40):
    """Split text into chunks of roughly max_chars characters, with a small overlap between chunks
    so an idea isn't cut in half at a boundary.
    """
    text = " ".join(text.split())  # normalize whitespace
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start = end - overlap  # step back a little so chunks overlap
    return chunks


def chunk_documents(documents):
    """Chunk every document and return a flat list of {"text": ..., "source": ...} dicts."""
    chunks = []
    for doc_id, doc in enumerate(documents, start=1):
        for chunk in chunk_text(doc):
            chunks.append({"text": chunk, "source": f"Document {doc_id}"})
    return chunks

chunks = chunk_documents(raw_documents)
print(f"Produced {len(chunks)} chunk(s):\n")
for c in chunks:
    print(f"[{c['source']}] {c['text']}")

Produced 3 chunk(s):

[Document 1] Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting from their first day. Unused days roll over up to a maximum of 5 days into the next year.
[Document 2] Expense Policy: Employees must submit expense reports within 30 days of purchase using the finance portal. Reports missing a receipt over $25 will be returned for correction.
[Document 3] Remote Work Policy: Remote employees must be reachable by chat or phone between 10am and 3pm in their local time zone, and are expected to attend the weekly all-hands meeting on video.


## Concept 3 — Embed and Store

We embed every chunk once and keep the vectors in a simple in-memory list of dictionaries. This *is* a vector
store — just a tiny, unindexed one. Chroma, FAISS, and Pinecone (Class 2) do this same job at a much larger
scale, with real indexing for speed.

In [6]:
def build_index(chunks):
    """Embed every chunk and return a list of {"text", "source", "embedding"} dicts -- our mini vector store."""
    index = []
    for c in chunks:
        index.append({
            "text": c["text"],
            "source": c["source"],
            "embedding": get_embedding(c["text"]),
        })
    return index

vector_store = build_index(chunks)
print(f"Indexed {len(vector_store)} chunk(s) into the in-memory vector store.")

Indexed 3 chunk(s) into the in-memory vector store.


## Concept 4 — Retrieve Top-k

Given a question, embed it and rank every stored chunk by cosine similarity. Keep the top `k` — usually 2 to 5
in a real system — as the context we'll hand to the LLM.

In [7]:
def retrieve(question, vector_store, k=2):
    """Return the top-k chunks (as dicts) from vector_store, ranked by cosine similarity to the question."""
    query_vec = get_embedding(question)
    scored = [
        (cosine_similarity(query_vec, item["embedding"]), item)
        for item in vector_store
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [item for _, item in scored[:k]]

question = "How many vacation days do new hires get?"
top_chunks = retrieve(question, vector_store, k=2)

for rank, chunk in enumerate(top_chunks, start=1):
    print(f"{rank}. [{chunk['source']}] {chunk['text']}")

1. [Document 1] Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting from their first day. Unused days roll over up to a maximum of 5 days into the next year.
2. [Document 2] Expense Policy: Employees must submit expense reports within 30 days of purchase using the finance portal. Reports missing a receipt over $25 will be returned for correction.


## Concept 5 — Construct a Grounded Prompt and Generate

Now we build the prompt exactly the way we did by hand in Class 1 — except the context is retrieved automatically
instead of pasted by us.

In [8]:
def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(f"[{c['source']}] {c['text']}" for c in retrieved_chunks)
    return f"""Answer using ONLY the context below. If the answer isn't in the context, say you don't know.

CONTEXT:
{context}

QUESTION: {question}
"""

prompt = build_prompt(question, top_chunks)
print(prompt)

Answer using ONLY the context below. If the answer isn't in the context, say you don't know.

CONTEXT:
[Document 1] Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting from their first day. Unused days roll over up to a maximum of 5 days into the next year.

[Document 2] Expense Policy: Employees must submit expense reports within 30 days of purchase using the finance portal. Reports missing a receipt over $25 will be returned for correction.

QUESTION: How many vacation days do new hires get?



In [9]:
answer = call_llm(prompt)
print(answer)

New hires accrue **15 paid vacation days per year**.


## Concept 6 — The Whole Pipeline, One Function

Let's wrap everything above into a single `rag_answer` function, so you can ask any question against this tiny
knowledge base in one call.

In [10]:
def rag_answer(question, vector_store, k=2):
    """Run the full retrieve -> construct prompt -> generate pipeline for a single question."""
    retrieved = retrieve(question, vector_store, k=k)
    prompt = build_prompt(question, retrieved)
    return call_llm(prompt), retrieved

if not GROQ_API_KEY:
    print("Skipping generation demo — set GROQ_API_KEY to generate an answer from retrieved chunks.")
    sources = retrieve("Do remote employees need to be on camera for meetings?", vector_store)
    print("SOURCES THAT WOULD BE USED:")
    for s in sources:
        print(f"  - [{s['source']}] {s['text'][:80]}...")
else:
    answer, sources = rag_answer("Do remote employees need to be on camera for meetings?", vector_store)
    print("ANSWER:", answer)
    print("\nSOURCES USED:")
    for s in sources:
        print(f"  - [{s['source']}] {s['text'][:80]}...")

ANSWER: Yes. Remote employees are expected to attend the weekly all‑hands meeting on video, which requires them to be on camera.

SOURCES USED:
  - [Document 3] Remote Work Policy: Remote employees must be reachable by chat or phone between ...
  - [Document 1] Vacation Policy: New hires accrue 15 paid vacation days per year, credited month...


## Limitations of This Simple Pipeline

- **Retrieval quality ceiling** — if the wrong chunk is retrieved, the answer will be wrong no matter how good the
  LLM is.
- **No built-in citations** — we printed sources manually above; a real product needs to surface this to users.
- **Stale index** — if `raw_documents` changes, you must re-run `build_index` or the answers go out of date.
- **Cost and latency stack up** — every question triggers one embedding call and one generation call.
- **Doesn't replace fine-tuning** — this pipeline is about facts, not tone or behavior.

Next steps: `class-3-langchain.ipynb` swaps this Python list for **Chroma** and **FAISS** and generates with Groq
through LangChain. Beyond that: re-ranking, hybrid (keyword + semantic) search, metadata filtering, real citations,
and systematic evaluation of retrieval and answer quality.

## Challenges

Each of these builds directly on `chunk_documents`, `build_index`, `retrieve`, and `rag_answer` above.

### Challenge 1 — Swap In Your Own Documents

Replace `raw_documents` with three short documents about a topic you know well (a hobby, a class, a project). Ask
a new question and print the answer along with which source(s) were used.

**Acceptance criteria:** your printed answer is correct according to your own documents, and cites the right
source.

In [ ]:
# TODO: write 3 new documents, rebuild the index, and ask a question about them


In [12]:
# TODO: modify or extend rag_answer to include inline citations in the returned text

# Original new documents (from the previous iteration of this cell's TODO)
documents_for_rag = [
    "Employees receive 15 days of paid vacation each year.",
    "Employees can work remotely two days per week.",
    "The company provides laptops to employees who need them for work."
]

# Define a function to create a basic vector store directly from documents
# This acts as a simpler version of chunk_documents + build_index for this small dataset
def create_simple_vector_store(docs):
    vector_store = []
    for i, doc_text in enumerate(docs):
        vector_store.append({
            "text": doc_text,
            "source": f"Document {i+1}", # Assign a simple source for citation
            "embedding": _embedder.encode(doc_text).tolist() # Use global _embedder
        })
    return vector_store

# Modified build_prompt to include citation instructions
def build_prompt_with_citations(question, retrieved_chunks):
    context_parts = []
    for chunk in retrieved_chunks:
        # Assuming chunk['source'] is like "Document N"
        context_parts.append(f"[{chunk['source']}] {chunk['text']}")

    context = "\n\n".join(context_parts)

    return f"""Answer the question concisely using ONLY the context below. If the answer isn't in the context, say you don't know.
For each statement in your answer, if it comes from a provided document, include the source citation at the end of the sentence or phrase like this: [Document N]. Do not make up citations.

CONTEXT:
{context}

QUESTION: {question}
"""

# New rag_answer function with citation support
def rag_answer_with_citations(question, docs, k=2):
    # Create vector store from the provided documents within this function's scope
    local_vector_store = create_simple_vector_store(docs)

    # Use the globally available retrieve function
    # Note: 'retrieve' expects the vector_store in a specific format which create_simple_vector_store provides
    retrieved = retrieve(question, local_vector_store, k=k)

    # Use our new prompt builder
    prompt = build_prompt_with_citations(question, retrieved)

    # Use the globally available call_llm function
    answer = call_llm(prompt)
    return answer, retrieved

# --- Demonstration of the new rag_answer_with_citations function ---

question_for_citations = "How many paid vacation days do employees receive and what are the remote work policies?"
answer_with_citations, sources_with_citations = rag_answer_with_citations(
    question_for_citations, documents_for_rag, k=3
)

print(f"\n--- Answer with Citations for: '{question_for_citations}' ---")
print("ANSWER:", answer_with_citations)
print("\nSOURCES USED:")
for s in sources_with_citations:
    print(f"  - [{s['source']}] {s['text']}")

Retrieved document:
Employees receive 15 days of paid vacation each year.


### Challenge 2 — Change the Chunk Size

Re-run `chunk_text` with a much smaller `max_chars` (e.g. 60) on one of the original documents and print the
resulting chunks. Then retrieve for the vacation-days question again using this smaller chunking and observe
whether the top retrieved chunk changes.

**Acceptance criteria:** you print the chunks produced by both chunk sizes and state in a comment whether
retrieval quality got better, worse, or stayed the same.

In [ ]:
# TODO: re-chunk with a smaller max_chars, rebuild the index, and compare retrieval results


In [14]:
# TODO: re-chunk with a smaller max_chars, rebuild the index,
# and compare retrieval results

def chunk_text(text, max_chars=50):
    return [text[i:i + max_chars] for i in range(0, len(text), max_chars)]


# Original chunks
original_chunks = documents

# Re-chunk with a smaller max_chars
small_chunks = []

for doc in documents:
    small_chunks.extend(chunk_text(doc, max_chars=50))


# Rebuild the index using the smaller chunks
small_embeddings = _embedder.encode(small_chunks)


# Use the same query as before
query = "How many paid vacation days do employees receive?"
query_embedding = _embedder.encode(query)


# Retrieve using the new smaller chunks
scores = cosine_similarity(
    [query_embedding],
    small_embeddings
)[0]

ranking = np.argsort(scores)[::-1]


# Print the new retrieval results
print("=== RETRIEVAL WITH SMALLER CHUNKS ===")

for i in ranking[:3]:
    print(f"Score: {scores[i]:.4f} | {small_chunks[i]}")


# Compare with the original chunks
print("\n=== ORIGINAL CHUNKS ===")

original_embeddings = _embedder.encode(original_chunks)

original_scores = cosine_similarity(
    [query_embedding],
    original_embeddings
)[0]

original_ranking = np.argsort(original_scores)[::-1]

for i in original_ranking[:3]:
    print(f"Score: {original_scores[i]:.4f} | {original_chunks[i]}")

=== RETRIEVAL WITH SMALLER CHUNKS ===
Score: 0.8418 | Employees receive 15 days of paid vacation each ye
Score: 0.5331 | Employees can work remotely two days per week.
Score: 0.3131 | The company provides laptops to employees who need

=== ORIGINAL CHUNKS ===
Score: 0.8439 | Employees receive 15 days of paid vacation each year.
Score: 0.5331 | Employees can work remotely two days per week.
Score: 0.3223 | The company provides laptops to employees who need them for work.


### Challenge 3 — Add a Real Citation

Modify `rag_answer` (or write a new version) so the returned answer string includes an inline citation like
`[Document 1]` after any sentence that uses that source's information.

**Acceptance criteria:** your function's output visibly includes at least one `[Document N]`-style citation.

In [ ]:
# TODO: modify or extend rag_answer to include inline citations in the returned text


In [17]:
# TODO: modify or extend rag_answer to include inline citations

def rag_answer_citations(question, documents):
    # Create embeddings for the documents
    doc_embeddings = _embedder.encode(documents)

    # Create embedding for the question
    query_embedding = _embedder.encode(question)

    # Calculate similarity
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np

    scores = cosine_similarity(
        [query_embedding],
        doc_embeddings
    )[0]

    # Get the top 2 relevant documents
    top_indices = np.argsort(scores)[::-1][:2]

    # Build context with source labels
    context = "\n\n".join(
        f"[Source {i+1}] {documents[index]}"
        for i, index in enumerate(top_indices)
    )

    # Create grounded prompt
    prompt = f"""
Answer the question using only the provided context.
Include an inline citation such as [Source 1] or [Source 2]
after each claim.

Context:
{context}

Question:
{question}
"""

    # Get the LLM answer
    answer = call_llm(prompt)

    return answer, [{"source": f"Source {i+1}", "text": documents[index]} for i, index in enumerate(top_indices)]


# Test it
question = "How many paid vacation days do employees receive?"

# Define documents_for_rag locally for self-containment
documents_for_rag = [
    "Employees receive 15 days of paid vacation each year.",
    "Employees can work remotely two days per week.",
    "The company provides laptops to employees who need them for work."
]

answer, sources = rag_answer_citations(
    question,
    documents_for_rag
)

print("ANSWER:", answer)
print("\nSOURCES USED:")
for s in sources:
    print(f"  - [{s['source']}] {s['text']}")

Employees receive 15 days of paid vacation each year【Source 1】.


### Challenge 4 — Force a "Don't Know"

Ask `rag_answer` a question that your three documents genuinely do not cover (e.g. something about a topic never
mentioned). Confirm the model says it doesn't know rather than guessing.

**Acceptance criteria:** you print the question and answer, and the answer explicitly states the information
isn't available, rather than fabricating a plausible-sounding response.

In [19]:
# TODO: ask a question your documents don't cover and confirm the model admits it doesn't know

question_unknown = "What is the company's policy on pet insurance?"

# Call the rag_answer function from cell 27bf4117 with the question that the documents don't cover
# It expects `vector_store` (a list of dictionaries) and returns both answer and sources.
answer_unknown, sources_unknown = rag_answer(question_unknown, vector_store)

print(f"QUESTION: {question_unknown}")
print(f"ANSWER: {answer_unknown}")
print("\nSOURCES USED:")
for s in sources_unknown:
    print(f"  - [{s['source']}] {s['text'][:80]}...")

# Confirm that the model admits it doesn't know
# The build_prompt function is designed to make the model say 'I don't know' if the answer isn't in the context.
# The expected output should reflect this, likely containing phrases like 'I don't know' or 'information isn't available'.
print("\n--- Confirmation ---")
if "don't know" in answer_unknown.lower() or "not in the context" in answer_unknown.lower() or "information isn't available" in answer_unknown.lower():
    print("The model successfully admitted it does not know the answer, as the information is not in the provided documents.")
else:
    print("The model did not explicitly state it doesn't know. Review the prompt or documents.")

QUESTION: What is the company's policy on pet insurance?
ANSWER: The provided documents do not contain any information about a pet‑insurance policy; neither the vacation policy nor the expense policy mentions pet insurance. [Source 1][Source 2]

--- Confirmation ---
The model did not explicitly state it doesn't know. Review the prompt or documents.


### Challenge 5 — Bonus: Add a Conflicting Document

Add a fourth document that directly contradicts one of the first three (e.g. a different, later "policy update"
stating a different number of vacation days). Ask the original question again and observe what the pipeline
retrieves and answers.

**Acceptance criteria:** you print what got retrieved and the final answer, and add a comment explaining, in your
own words, why this is a real limitation of simple RAG systems.

In [ ]:
# TODO: add a conflicting fourth document, re-run the pipeline, and comment on what happens


In [22]:
# TODO: add a conflicting fourth document, re-run the pipeline,
# and comment on what happens

# Initialize documents with the original raw_documents
documents = raw_documents.copy()

# Add a conflicting policy update
documents.append(
    "Policy Update: Starting this year, employees receive 20 days of paid vacation each year."
)

# Rebuild the embeddings/index
doc_embeddings = _embedder.encode(documents)

# Ask the original question
question = "How many paid vacation days do employees receive?"

# Create query embedding
query_embedding = _embedder.encode(question)

# Calculate similarity
scores = cosine_similarity(
    [query_embedding],
    doc_embeddings
)[0]

# Retrieve the top 2 documents
top_indices = np.argsort(scores)[::-1][:2]

# Print what was retrieved
print("=== RETRIEVED DOCUMENTS ===")

for i, index in enumerate(top_indices):
    print(f"[Source {i+1}] Score: {scores[index]:.4f}")
    print(documents[index])
    print()

# Build context for the LLM
context = "\n\n".join(
    f"[Source {i+1}] {documents[index]}"
    for i, index in enumerate(top_indices)
)

# Ask the LLM to answer using the retrieved context
prompt = f"""
Answer the question using the provided context.
If there are conflicting policies, mention the conflict rather than
pretending there is only one answer.

Context:
{context}

Question:
{question}
"""

answer = call_llm(prompt)

print("=== FINAL ANSWER ===")
print(answer)


# Simple RAG limitation:
# A simple RAG system retrieves documents based on similarity, but it does
# not automatically know which conflicting document is newer or authoritative.
# Therefore, it may retrieve an outdated policy, a new policy, or both and
# potentially give an incorrect or ambiguous answer.

=== RETRIEVED DOCUMENTS ===
[Source 1] Score: 0.8239
Policy Update: Starting this year, employees receive 20 days of paid vacation each year.

[Source 2] Score: 0.7931
Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting
    from their first day. Unused days roll over up to a maximum of 5 days into the next year.

=== FINAL ANSWER ===
The two sources give different figures:

- **Source 1** states that, starting this year, **all employees receive 20 paid vacation days each year**.  
- **Source 2** describes a separate rule for **new hires**, who **accrue 15 paid vacation days per year** (credited monthly), with unused days rolling over up to 5 days.

Because these policies conflict—one says 20 days for employees generally, the other says 15 days for new hires—the exact amount an employee receives depends on which policy applies. If both policies are in effect, existing employees would get 20 days, while new hires would accrue 15 days per year.


### Observation and Limitation:

In this scenario, after adding a conflicting document, the RAG pipeline might retrieve *both* conflicting documents if their embeddings are similar enough to the query. The LLM then receives both pieces of information in the context. Depending on the prompt and the LLM's behavior, it might:

1.  **State both facts:** It might present both "15 paid vacation days" and "20 paid vacation days" without resolving the conflict, or try to reconcile them if it detects a temporal aspect (e.g., "originally 15, now 20").
2.  **Prioritize one:** It might implicitly prioritize one over the other based on phrasing (e.g., "Effective immediately" might be weighted more heavily if the LLM understands temporal cues).
3.  **Confuse the answer:** It might provide a mixed or ambiguous answer.

This highlights a key limitation of simple RAG systems: **they are not inherently designed to resolve conflicting information within their knowledge base.** If conflicting documents exist, a simple RAG pipeline will simply present whatever relevant information it retrieves to the LLM. Production-grade RAG systems often employ strategies like:

*   **Version control:** Keeping track of document versions and retrieving only the most recent/authoritative one.
*   **Document ranking/scoring:** Assigning confidence scores or trust levels to sources.
*   **Contradiction detection:** Explicitly trying to identify and flag conflicting information during retrieval or generation.
*   **Human-in-the-loop:** Allowing human oversight to resolve conflicts.

In [23]:
# Limitation of simple RAG: when two documents contain conflicting information,
# the retriever may return both documents because both are relevant to the query.
# The LLM may choose one, report both, or give an ambiguous answer because simple
# RAG does not automatically know which document is newer or more trustworthy.
# This shows why production RAG systems need things like source ranking, versioning,
# or conflict detection.